In [ ]:
import logging
import os
import pathlib
import dotenv
import psycopg
import pandas as pd
import sqlalchemy as sa

logging.basicConfig(format="%(asctime)s - %(levelname)s - %(message)s", level=logging.INFO)

OUTDIR = "/code/Users/rocrate/omop"


In [ ]:
!df

In [ ]:
user = dotenv.get("POSTGRES_USER")
password = dotenv.get("POSTGRES_PASSWORD")
host = dotenv.get("POSTGRES_HOST")

In [ ]:
def get_sql(query, **kwargs):
    with psycopg.connect(f"postgres://{user}:{password}@{host}:5432") as conn:
    # with sa.create_engine(f"postgres+psycopg://{user}:{password}@{host}:5432") as conn:
        return pd.read_sql(query, conn, **kwargs)

In [ ]:
# get_sql("select schema_name from information_schema.schemata order by 1")
# get_sql("select pg_size_pretty(pg_database_size('aimahead'))")

query = """
select table_schema, table_name, 
pg_size_pretty(pg_table_size(table_schema||'.'||table_name)) 
from information_schema.tables 
where table_schema='omopcdmv2' 
order by 1,2,3"""

df_tables = get_sql(query)

pathlib.Path(f"{OUTDIR}").mkdir(parents=True, exist_ok=True)
for idx, row in df_tables.iterrows():
    logging.info(f"reading {row['table_name']}")
    dfs = get_sql(f"select * from omopcdm.{row['table_name']} limit 10000000", chunksize=int(10e6))
    for idx, df in enumerate(dfs):
        fp = pathlib.Path(f"{OUTDIR}/{row['table_name']}.{idx}.parquet")
        fp.parent.mkdir(parents=True, exist_ok=True)
        logging.info(f"writing {fp}")
        df.to_parquet(fp, compression="zstd", index=False)
        logging.info(f"reading next batch..")
logging.info(f"done")

In [ ]:
# get_sql("select schema_name from information_schema.schemata order by 1")
# get_sql("select pg_size_pretty(pg_database_size('aimahead'))")

query = """
select table_schema, table_name, 
pg_size_pretty(pg_table_size(table_schema||'.'||table_name)) 
from information_schema.tables 
where table_schema='omopcdmv2' 
order by 1,2,3"""

df_tables = get_sql(query)

for idx, row in df_tables.iterrows():
    logging.info(f"reading {row['table_name']}")
    df = get_sql(f"select * from omopcdm.{row['table_name']} limit 10000")
    fp = pathlib.Path(f"{OUTDIR}/{row['table_name']}.tsv")
    fp.parent.mkdir(parents=True, exist_ok=True)
    logging.info(f"writing {fp}")
    df.to_csv(fp, sep='\t', index=False)
    # break
logging.info(f"done")


In [ ]:
# get_sql("select schema_name from information_schema.schemata order by 1")
# get_sql("select pg_size_pretty(pg_database_size('aimahead'))")

query = """
select table_schema, table_name, 
pg_size_pretty(pg_table_size(table_schema||'.'||table_name)) 
from information_schema.tables 
where table_schema='omopcdmv2' 
order by 1,2,3"""

df_tables = get_sql(query)
dfs = []
for idx, row in df_tables.iterrows():
    logging.info(f"reading {row['table_name']}")
    dfs.append(get_sql(f"select '{row['table_name']}', count(*) from omopcdm.{row['table_name']}"))
    # break
logging.info(f"done")


In [ ]:
pd.concat(dfs)

In [ ]:
query = """
select count(*) from omopcdm.concept
"""
get_sql(query)

In [ ]:
query = """
select table_schema, table_name, pg_table_size('omopcdm'||'.'||table_name),
pg_size_pretty(pg_table_size('omopcdm'||'.'||table_name)) 
from information_schema.tables 
where table_schema='omopcdmv2' 
order by 1,2,3"""
get_sql(query)

In [ ]:
query = """
select count(*) from omopcdmv2.concept
"""
get_sql(query)

In [ ]:
query = """
select relname, n_live_tup
from pg_stat_user_tables
where schemaname='omopcdmv2' 
order by 1"""
get_sql(query)

In [ ]:
output = query_executor(query_tables, 'all')